# 16 — Video Generation Model Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Video Prompt Structure

In [3]:
def build_video_prompt(subject, action, camera_motion, setting, style, duration="5 seconds"):
    return (
        f"{subject} {action}, {camera_motion}, {setting}, "
        f"{style}, {duration} video clip"
    )

example = build_video_prompt(
    subject="a golden retriever",
    action="running through tall grass, tongue out, joyful",
    camera_motion="tracking shot following the dog, slight camera shake",
    setting="sunny meadow, wildflowers, warm afternoon light",
    style="cinematic, 4K, shallow depth of field",
    duration="6 seconds"
)
print("Video prompt:")
print(example)


Video prompt:
a golden retriever running through tall grass, tongue out, joyful, tracking shot following the dog, slight camera shake, sunny meadow, wildflowers, warm afternoon light, cinematic, 4K, shallow depth of field, 6 seconds video clip


## Camera Movement Vocabulary

In [4]:
camera_moves = {
    "Pan":              "camera rotates left/right on fixed axis",
    "Tilt":             "camera rotates up/down on fixed axis",
    "Dolly/Track":      "camera physically moves forward/backward/sideways",
    "Zoom":             "focal length changes, subject appears larger/smaller",
    "Crane/Jib":        "camera moves vertically, sweeping shots",
    "Handheld":         "intentional slight shake, documentary feel",
    "Steadicam":        "smooth movement while moving through space",
    "Aerial/Drone":     "bird's eye view, moving overhead",
    "Pull-back reveal": "starts close, pulls back to reveal context",
}
print("Camera movement terms for video prompts:")
for move, desc in camera_moves.items():
    print(f"  {move:20s}: {desc}")


Camera movement terms for video prompts:
  Pan                 : camera rotates left/right on fixed axis
  Tilt                : camera rotates up/down on fixed axis
  Dolly/Track         : camera physically moves forward/backward/sideways
  Zoom                : focal length changes, subject appears larger/smaller
  Crane/Jib           : camera moves vertically, sweeping shots
  Handheld            : intentional slight shake, documentary feel
  Steadicam           : smooth movement while moving through space
  Aerial/Drone        : bird's eye view, moving overhead
  Pull-back reveal    : starts close, pulls back to reveal context


## LLM-Assisted Video Prompt Writing

In [5]:
scene_idea = "a scientist makes an exciting discovery in a lab"

video_prompt_gen = (
    f"Write a detailed video generation prompt for this scene:\n\"{scene_idea}\"\n\n"
    "Include:\n"
    "1. Subject and action (specific, vivid)\n"
    "2. Camera movement (cinematic technique)\n"
    "3. Setting and atmosphere\n"
    "4. Lighting and color palette\n"
    "5. Visual style and mood\n"
    "6. Duration (5-10 seconds)\n\n"
    "Format as a single paragraph prompt under 100 words."
)
result = chat([{"role":"user","content":video_prompt_gen}], temperature=0.7, max_tokens=200)
print(result.strip())


"Create a 7-second video of a scientist, Dr. Maria Rodriguez, making an exciting discovery in a lab. Camera movement: a slow-motion zoom (0.5x speed) from a wide shot of the lab to a close-up of Dr. Rodriguez's face, her eyes widening with excitement as she holds up a petri dish containing a glowing, iridescent sample. Setting: a dimly lit lab with rows of humming machinery, a faint scent of disinfectant in the air. Lighting: warm, soft fluorescent lights. Color palette: cool blues and greens. Visual style: cinematic, with a slight grain and depth of field. Mood: suspenseful and triumphant."


## Scene-to-Scene Continuity

In [6]:
story = "A time-lapse of a sunflower growing from seed to full bloom"

continuity_prompt = (
    f"Write 3 sequential video prompts for this story: \"{story}\"\n"
    "Each prompt should be 5 seconds.\n"
    "Maintain consistent: lighting style, color palette, camera position.\n"
    "Label each: [Scene 1], [Scene 2], [Scene 3]."
)
result = chat([{"role":"user","content":continuity_prompt}], temperature=0.5, max_tokens=300)
print(result.strip())


[Scene 1]
(0s-5s)
- Camera position: Overhead, 3 feet above the soil
- Lighting style: Soft, natural light with a slight warm tone
- Color palette: Vibrant greens and yellows
- Prompt: "A single sunflower seed is planted in the soil. The camera captures the initial stages of growth, with the seed beginning to sprout a small green stem."

[Scene 2]
(5s-10s)
- Camera position: Same as Scene 1
- Lighting style: Soft, natural light with a slight warm tone
- Color palette: Vibrant greens and yellows
- Prompt: "The sunflower stem grows taller and begins to develop its first set of leaves. The camera captures the rapid growth and development of the plant."

[Scene 3]
(10s-15s)
- Camera position: Same as Scene 1
- Lighting style: Soft, natural light with a slight warm tone
- Color palette: Vibrant greens and yellows
- Prompt: "The sunflower continues to grow, its stem reaching towards the sky. The camera captures the moment when the first sunflower petals begin to bloom, revealing the bright y

## Negative Prompts for Video

In [7]:
pos_video = "slow motion ocean waves crashing on rocky coast, golden sunset, cinematic"
neg_video = "fast cuts, text overlays, watermark, digital artifacts, morphing faces, flickering"

explain = chat([{"role":"user","content":f"Explain why these negative video prompt terms matter:\n{neg_video}\nOne line per term."}],
               temperature=0, max_tokens=150)
print("Positive:", pos_video)
print()
print("Negative:", neg_video)
print()
print("Why negatives matter:")
print(explain.strip())


Positive: slow motion ocean waves crashing on rocky coast, golden sunset, cinematic

Negative: fast cuts, text overlays, watermark, digital artifacts, morphing faces, flickering

Why negatives matter:
Here are one-line explanations for each term:

1. **Fast cuts**: Fast cuts can make a video appear jarring, disorienting, or even nauseating, which can negatively impact the viewer's experience and engagement.
2. **Text overlays**: Overusing text overlays can clutter the screen, distract from the content, and make the video appear amateurish or unprofessional.
3. **Watermark**: A visible watermark can be distracting, unprofessional, and may even be considered a form of copyright infringement, which can harm the creator's reputation.
4. **Digital artifacts**: Digital artifacts, such as compression artifacts or aliasing, can make a video appear low-quality, outdated, or even fake, which can undermine the creator's credibility.
